In [1]:
# Load pandas.
import pandas as pd

In [2]:
# Load the raw orders file into a dataframe we can clean.
data = pd.read_csv('orders.csv')

orders = pd.DataFrame(data)
df_orders = orders.copy()

1. Structure (Shape & Data Type)

In [3]:
# Check the shape and column types.
df_orders.info()
df_orders.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  object 
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 6.9+ MB


(226909, 4)

2. Primary key integrity

In [4]:
# Check for fully duplicate rows.
df_orders.duplicated().sum()

np.int64(0)

In [5]:
# Check if any order_id values repeat (they shouldn't).
df_orders['order_id'].duplicated().sum()

np.int64(0)

3. Data Types

In [6]:
# Convert created_date to a real date type.
df_orders['created_date'] = pd.to_datetime(df_orders['created_date'], errors='coerce')
#df_orders['order_id'] = df_orders['order_id'].astype('int64')
#df_orders['total_paid'] = df_orders['total_paid'].astype('int64') # to change dtype
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      226909 non-null  int64         
 1   created_date  226909 non-null  datetime64[ns]
 2   total_paid    226904 non-null  float64       
 3   state         226909 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 6.9+ MB


4. Missing values: total_paid is NaN when state is 'Pending'. But there are many orders that have 'Pending' state but haven't NaN value for total_paid.

In [7]:
# Check for missing values in each column.
df_orders.isna().sum()

order_id        0
created_date    0
total_paid      5
state           0
dtype: int64

In [8]:
# Look at the rows where total_paid is missing.
df_orders.loc[df_orders['total_paid'].isna(), ['total_paid', 'state', 'created_date']]

,total_paid,state,created_date
127701,NaN,Pending,2017-11-20 18:54:39
132013,NaN,Pending,2017-11-22 12:15:24
147316,NaN,Pending,2017-11-27 10:32:37
148833,NaN,Pending,2017-11-27 18:54:15
149434,NaN,Pending,2017-11-27 21:52:08


Drop Nan values in 'total_paid' column

In [9]:
# See what share of rows are missing total_paid.
df_orders['total_paid'].isna().value_counts(normalize =  True)

total_paid
False    0.999978
True     0.000022
Name: proportion, dtype: float64

In [10]:
# Drop the rows with missing total_paid -- it's a tiny fraction.
df_orders = df_orders.dropna(axis = 0)

In [11]:
# Check the columns after dropping them.
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
Index: 226904 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      226904 non-null  int64         
 1   created_date  226904 non-null  datetime64[ns]
 2   total_paid    226904 non-null  float64       
 3   state         226904 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 8.7+ MB


5. Categorical Validation: Typos, stray casing, and trailing whitespace hide in text categories - "Completed" vs "completed " silently splits one group into two in any groupby.



In [12]:
# Trim stray spaces from the state column and list the unique values.
df_orders['state'] = df_orders['state'].str.strip()
sorted(df_orders['state'].unique())


['Cancelled', 'Completed', 'Pending', 'Place Order', 'Shopping Basket']

6. Numeric range & Outliers : describe() surfaces implausible values in one call - a negative total, or an outlier cluster tied to one state, before you'd otherwise spot it row by row.

In [13]:
# Check for negative totals.
#df_orders.describe()
(df_orders['total_paid'] < 0).sum()

np.int64(0)

In [14]:
# Save the cleaned orders to a new CSV file.
df_orders.to_csv('orders.clean.csv')